# RSA phase estimation

Inspect the modular exponentiation and Fourier readout used in factoring. The [worked modular exponentiation example](register_factoring_via_modexp.ipynb) shows the classical period and factor extraction for 221.

Adapted from the Qualtran v0.7.0 RSA tutorial. Copyright 2024 Google LLC. Licensed under Apache-2.0.

In [ ]:
from collections import Counter

from IPython.display import Markdown, SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.modular_exponentiation import ModularExponentiation
from flagquantum.algorithms.primitives.rsa_phase_estimation import RSAPhaseEstimation
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.dynamic import DynamicCircuit

def show_calls(graph, title=str):
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls, title=title))))
    display(Markdown(format_call_graph_markdown(graph.root, graph.direct_calls, graph.leaf_counts, title=title)))

## Modular exponentiation

The exponent passes through, and a fresh output word holds the modular power. The initializer prepares 1. The controlled multiplier walks the exponent from low to high significance and squares the constant after each bit.

In [ ]:
g, N, n_e, n_x = symbols("g N n_e n_x")
symbolic_modexp = ModularExponentiation(g, N, n_e, n_x)
small_exponent = ModularExponentiation(4, 15, 3, 2048)
factoring_modexp = ModularExponentiation.for_shor(13 * 17, 9)
display(SVG(field_signature_svg((
    ("ModExp(4,15)", small_exponent.source_signature, {}),
    ("ModExp(9,221)", factoring_modexp.source_signature, {}),
    ("ModExp(g,N)", symbolic_modexp.source_signature, {}),
))))

In [ ]:
graph = small_exponent.source_call_graph()
assert graph.leaf_counts == {"CModMulK": 3, "|1>": 1}
show_calls(graph)

## RSA phase estimation

The source declares an empty signature because it owns the phase and target registers. The native binding receives fresh physical wires and result slots. It prepares `2n` phase qubits, computes the modular power, and measures the forward QFT in the original physical order.

The locked call graph declares an `n`-wire MeasureQFT child, while the executable body passes `2n` wires. The graph below preserves the declared width. The emitted readout uses the body width. The modular target is dirty-released in the source and stays available on caller-owned native wires.

[Circuit construction reference](https://arxiv.org/abs/quant-ph/0205095)

In [ ]:
n, p, g = symbols("n p g")
phase_symbolic = RSAPhaseEstimation(n, p, g)
phase_221 = RSAPhaseEstimation.for_shor(13 * 17, 9)
assert phase_221.phase_width == 16
display(SVG(field_signature_svg((
    ("RSAPhaseEstimate(221)", phase_221.source_signature, {}),
    ("RSAPhaseEstimate(p)", phase_symbolic.source_signature, {}),
))))

In [ ]:
calls = phase_221.source_call_graph()
assert calls.leaf_counts == {("MeasureQFT", 8): 1, ("CModMulK", 8): 16}
show_calls(calls, title=lambda call: f"{call[0]}({call[1]})")

## Bind and inspect a small circuit

For base 2 and modulus 3, four phase wires receive the Fourier readout in the supplied slot order. The result register can remain correlated with those bits. See the [audit](../../docs/development/RSA_PHASE_FILE_AUDIT.md) for the complete projected complex amplitudes and workspace checks.

In [ ]:
phase = RSAPhaseEstimation.for_shor(3, 2)
q, c = phase.workspace_requirements()
circuit = DynamicCircuit(6 + q).append(phase.make_on(
    phase=(0, 1, 2, 3), target=(4, 5), workspace=tuple(range(6, 6 + q)),
    classical_bits=tuple(range(c)), result_bits=tuple(range(c, c + 4)),
))
ir = circuit.to_ir()
readout = [(gate.wires[0], gate.metadata["classical_bit"]) for gate in ir.instructions[-4:]]
assert readout == list(zip(range(4), range(c, c + 4), strict=True))
{"wires": ir.n_wires, "emitted": dict(Counter(gate.name for gate in ir.instructions))}